#**1. BIBLIOTECAS**

In [1]:
import json
import time
import gspread
import requests
import datetime
import pandas as pd
import urllib.request
# from pyspark.sql import SparkSession # Removed Spark import
#from pyspark.sql.functions import col # Removed Spark import
from concurrent.futures import ThreadPoolExecutor
from gspread_dataframe import get_as_dataframe, set_with_dataframe
#from pyspark.sql.types import StructType, StructField, StringType, FloatType # Removed Spark import

# **2. DADOS DE ACESSO**

##**2.1. Credenciais**

In [2]:
credentials = {
        "username": "wisllaynni.silva@samprojetos.com",
        "password": "eioa85lq"
    }

## **2.2. URL's**

In [3]:
url_base = "https://api.s360web.com"
url_login = url_base + "/api/login"
url_samplelist = url_base + "/api/v1/amostra/list"
url_sampleresults = url_base + "/api/v1/sampleResult/search"

##**2.3. Token**

In [4]:
try:
    url = url_login

    hdr = {
        'Content-Type': 'application/json',
    }

    credentials = credentials

    data = json.dumps(credentials)
    req = urllib.request.Request(url, headers=hdr, data=data.encode("utf-8"), method='POST')

    with urllib.request.urlopen(req) as response:
        response_data = response.read().decode("utf-8")
        response_json = json.loads(response_data)

        access_token = response_json.get("access_token")

        print("Token:", access_token)

except Exception as e:
    print("Erro:", e)

Token: eyJhbGciOiJIUzI1NiJ9.eyJwcmluY2lwYWwiOiJINHNJQUFBQUFBQUFBS1ZVUFc4VFFSRGRHRUtRSXBHRVFGQ0tJRVNnZzdORitCQ0VnaU4yd05FNVo5M1pGQUZocmNcL3J5NGE5M1dOM0w0NHBVQ3FRU0FFQ0lvRW9LUkFTXC80RWlKVFQ4QUFRRmJTb0tXbWJQc1oxVUVXS3IxY3piTnpOdlp1ZlROaHBVRXAwT1JHUmhwa0ltNnBoWk9DQktDYXZVcmlvaTgwUmp5dFRuazcrdjNcL3Y0N0hjR0hWcEM0NVJyRWtvY1lMRWdsbm1lRUVuMldFczBrRUtKcG5iUWNDQmtXZElJUzRvMU91cXM0RldjWlppSFdWOUx5c05aQngxdUpvd3Q0b2c4UklcL1JnSU5HRThtS0VRNUpOSTlYYVNENGpuMmlaM2R3V3lUYXAwcVRDS2ZlTllsbUlEWmthc1VzQ1NtM1ZHem9GUWtTU1hYYlNxQ1dScWNXNjFZS05OV2h6aG5JR1A0TWJlek4wSzJ2a0VEUEF2Y0ZJY01keHFhRVRGdENQckI2M0ZBaTJST2dUNTI1bGtGRFMyZ01CNEZJdUY0VXZMQVdVMGthUzJpMGIzTkU4TUNZamdmZ0lWeFRhTVp1NkJEaHVNNUlBK1RFaVY0V0VKVVNwZEZJSjlsRVU1YjFpVFpheGxncHlLNnhvOWxoa3hidmFyc1dEOENCanA4MTd5enp6cG9UakVHVlZIQjFwc29qMGFCTmFvSUIzXC9yVXkyXC9QMzYxWE13aUJCdWYyZjlPM1Q5NUU2MVwvdVwvem1aQ2pzUWFEU3hLOVUrYkhZdGhtekcrc3dWU1V6azcyXC9LcnphM245NDlBSkVOWXY3ZjlUOWo3eWpWbmhOUmpDWFdZbGRQZ0xaMUVPNW1cL0dcL3VUOTVWdlczNU5Jb1pnUW1DWVdcLzBRdlNKb2R5RFVyQ3UzaHFkOEZ5

##**2.4. Header**

In [5]:
header = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

##**2.5. Sheets**

###**2.5.1. Autenticação no Sheets**

In [6]:
from google.colab import auth
from google.auth import default

# Autentica no Google
auth.authenticate_user()

# Use 'default' para pegar as credenciais no formato esperado pelo gspread
creds, _ = default()
gc = gspread.authorize(creds)

#**3. STATUS DE AMOSTRAS**

##**3.1. Execução**

In [6]:
url = url_samplelist
hdr = header

# Função para buscar uma página
def fetch_pagina(pagina):
    try:
        filters = {
            "maximoPorPagina": 100,
            "numeroPagina": pagina,
            "situacao": ["EM_PROCESSAMENTO", "COLETADA", "SEGREGADA", "FINALIZADA"]
        }
        data = json.dumps(filters).encode("utf-8")
        req = urllib.request.Request(url, headers=hdr, data=data)
        req.get_method = lambda: 'POST'
        with urllib.request.urlopen(req) as response:
            result = json.loads(response.read().decode('utf-8'))
            return result.get("resultados", [])
    except Exception as e:
        print(f"Erro na página {pagina}:", e)
        return []

try:
    filters = {
        "maximoPorPagina": 100,
        "numeroPagina": 1,
        "situacao": ["EM_PROCESSAMENTO", "COLETADA", "SEGREGADA", "FINALIZADA"]
    }
    data = json.dumps(filters).encode("utf-8")
    req = urllib.request.Request(url, headers=hdr, data=data)
    req.get_method = lambda: 'POST'
    with urllib.request.urlopen(req) as response:
        primeira_resposta = json.loads(response.read().decode('utf-8'))
        total_paginas = primeira_resposta.get("totalPaginas", 1)
        resultados_totais = primeira_resposta.get("resultados", [])

    # Buscar as páginas restantes em paralelo
    paginas_restantes = list(range(2, total_paginas + 1))
    with ThreadPoolExecutor(max_workers=5) as executor:
        resultados = executor.map(fetch_pagina, paginas_restantes)

    for pagina_resultado in resultados:
        resultados_totais.extend(pagina_resultado)

    print(f"Total de resultados: {len(resultados_totais)}")
    print(json.dumps(resultados_totais, indent=2))

except Exception as e:
    print("Erro:", e)


Total de resultados: 2465
[
  {
    "numeroAmostra": "230017261",
    "codigoExterno": "M0202HD-M-6RT001",
    "situacao": "COLETADA",
    "foiSegregado": false,
    "tipoColeta": "EQUIPAMENTO",
    "compartimento": {
      "id": 1388964,
      "nome": "UNIDADE HIDR\u00c1ULICA"
    },
    "equipamento": {
      "id": 388042,
      "modelo": "UNIDADE HIDR\u00c1ULICA SHAFER",
      "chassiSerie": "G02-14HD201",
      "frota": "G02-14HD201",
      "codigoExterno": "334"
    },
    "cliente": {
      "id": 6845,
      "nome": "SAMARCO MINERACAO - MATIPO MG"
    },
    "obra": {
      "id": 24968,
      "nome": "SAMARCO MINERACAO - MATIPO MG"
    },
    "area": "MINERODUTO MATIP\u00d3",
    "setor": "ESTA\u00c7\u00c3O DE BOMBA 5"
  },
  {
    "numeroAmostra": "G02CP014",
    "codigoExterno": "G02-CP014",
    "situacao": "COLETADA",
    "foiSegregado": false,
    "tipoColeta": "AVULSA",
    "cliente": {
      "id": 6844,
      "nome": "SAMARCO MINERACAO - GERMANO - MARIANA MG"
    },
    "ob

##**3.2.DataFrame**

In [7]:
# Transforma os resultados em um DataFrame expandindo os campos aninhados
df = pd.json_normalize(
    resultados_totais,
    sep='_',  # para gerar colunas como cliente_nome, obra_nome etc.
    max_level=2  # expande até 2 níveis; aumente se necessário
)

# Visualiza as colunas
# print("Colunas disponíveis:", df.columns.tolist())

tb_status = pd.DataFrame(df)

display(tb_status)

,numeroAmostra,codigoExterno,situacao,foiSegregado,tipoColeta,area,setor,compartimento_id,compartimento_nome,equipamento_id,...,equipamento_chassiSerie,equipamento_frota,equipamento_codigoExterno,cliente_id,cliente_nome,obra_id,obra_nome,responsavelRegistro,dataDesegregacao,dataFinalizacao
0,230017261,M0202HD-M-6RT001,COLETADA,False,EQUIPAMENTO,MINERODUTO MATIPÓ,ESTAÇÃO DE BOMBA 5,1388964.0,UNIDADE HIDRÁULICA,388042.0,...,G02-14HD201,G02-14HD201,334,6845,SAMARCO MINERACAO - MATIPO MG,24968.0,SAMARCO MINERACAO - MATIPO MG,NaN,NaN,NaN
1,G02CP014,G02-CP014,COLETADA,False,AVULSA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,6844,SAMARCO MINERACAO - GERMANO - MARIANA MG,24967.0,SAMARCO MINERACAO - GERMANO - MARIANA MG,NaN,NaN,NaN
2,2400459680,10708,COLETADA,False,EQUIPAMENTO,FORNO,NaN,1811715.0,MANCAL LOA ROTOR - DRENO,395114.0,...,U04-06VT002,U04-06VT002,NaN,6847,SAMARCO MINERACAO - UBU - ANCHIETA ES,24970.0,SAMARCO MINERACAO - UBU - ANCHIETA ES,THAMIRIS SILVA - NEXUX (ATIVA OK),NaN,NaN
3,17CP014,g02-17cp014,COLETADA,False,AVULSA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,6844,SAMARCO MINERACAO - GERMANO - MARIANA MG,24967.0,SAMARCO MINERACAO - GERMANO - MARIANA MG,NaN,NaN,NaN
4,230011756,G02-17CP013,COLETADA,False,AVULSA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,6844,SAMARCO MINERACAO - GERMANO - MARIANA MG,24967.0,SAMARCO MINERACAO - GERMANO - MARIANA MG,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2460,2500364899,NaN,FINALIZADA,False,EQUIPAMENTO,FORNO,NaN,2920317.0,MANCAL ROTOR LOA - DRENO,990613.0,...,U03-06VT003,U03-06VT003,NaN,6847,SAMARCO MINERACAO - UBU - ANCHIETA ES,24970.0,SAMARCO MINERACAO - UBU - ANCHIETA ES,THAMIRIS SILVA - NEXUX (ATIVA OK),NaN,2025-08-29T16:05:35.580Z
2461,2500398107,NaN,FINALIZADA,False,EQUIPAMENTO,FORNO,NaN,3065569.0,MOTOR LA - DRENO,990613.0,...,U03-06VT003,U03-06VT003,NaN,6847,SAMARCO MINERACAO - UBU - ANCHIETA ES,24970.0,SAMARCO MINERACAO - UBU - ANCHIETA ES,THAMIRIS SILVA - NEXUX (ATIVA OK),NaN,2025-08-29T16:00:41.059Z
2462,2500463853,NaN,FINALIZADA,False,EQUIPAMENTO,FORNO,NaN,3065569.0,MOTOR LA - DRENO,990613.0,...,U03-06VT003,U03-06VT003,NaN,6847,SAMARCO MINERACAO - UBU - ANCHIETA ES,24970.0,SAMARCO MINERACAO - UBU - ANCHIETA ES,THAMIRIS SILVA - NEXUX (ATIVA OK),NaN,2025-08-29T15:57:24.427Z
2463,2500364878,NaN,FINALIZADA,False,EQUIPAMENTO,NaN,NaN,1814863.0,MANCAL LA ROTOR - DRENO,529362.0,...,U04-06VT001B,U04-06VT001B,NaN,6847,SAMARCO MINERACAO - UBU - ANCHIETA ES,24970.0,SAMARCO MINERACAO - UBU - ANCHIETA ES,THAMIRIS SILVA - NEXUX (ATIVA OK),NaN,2025-08-29T15:56:18.016Z


##**3.3.Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_lub_ext_als_status"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, tb_status)

print("Dados enviados com sucesso para o Google Sheets!")

Dados enviados com sucesso para o Google Sheets!


#**4. RESULTADO DE AMOSTRAS**

##**4.1. Execução**

In [7]:
# Função para fazer a requisição paginada
def buscar_dados(offset):
    url = url_sampleresults
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    ontem = (datetime.date.today() - datetime.timedelta(days=1)).isoformat()


    filters = {
        "sinceResultDate": ontem,
        "untilResultDate": ontem,
        "offset": offset,
        "max": 50,
        "order": "resultDate",
        "sort": "desc"
    }
    data = json.dumps(filters).encode("utf-8")
    req = urllib.request.Request(url, headers=headers, data=data)
    req.get_method = lambda: 'POST'

    try:
        response = urllib.request.urlopen(req)
        result = json.loads(response.read())
        return result.get("results", [])
    except Exception as e:
        print("Erro ao requisitar dados:", e)
        return []

# Função de transformação para expandir testResults
def processar_linha(linha):
    validResult = linha.get("validResult") or {}
    equipment = linha.get("equipment") or {}
    collectionData = linha.get("collectionData") or {}
    oil = collectionData.get("oil") or {}
    compartment = linha.get("compartment") or {}
    site = equipment.get("site") or {}
    family = equipment.get("family") or {}
    maker = equipment.get("maker") or {}
    oil_viscosity = oil.get("viscosity") or {}
    oil_manufacturer = oil.get("manufacturer") or {}
    compartment_type = compartment.get("type") or {}

    base = {
        "sampleNumber": linha.get("sampleNumber"),
        "evaluation": validResult.get("evaluation"),
        "inspectionAction": validResult.get("inspectionAction"),
        "resultStatus": validResult.get("resultStatus") or "sampleResultsStatus",
        "equipmentTag": equipment.get("tag"),
        "equipmentFamily": family.get("name"),
        "equipmentMaker": maker.get("name"),
        "equipmentModel": equipment.get("model"),
        "equipmentSite": site.get("name"),
        "equipmentArea": equipment.get("area"),
        "equipmentSector": equipment.get("sector"),
        "compartmentName": compartment.get("name"),
        "compartmentTypeName": compartment_type.get("name"),
        "registrationDate": collectionData.get("registrationDate"),
        "dateSampled": collectionData.get("dateSampled"),
        "oilViscosity": oil_viscosity.get("name"),
        "oilType": oil_manufacturer.get("name"),
        "receiptDate": linha.get("receiptDate"),
        "resultDate": linha.get("resultDate")
    }

    # ── Pivota testResults em colunas ────────────────────────────────────────
    # Cada teste vira: {testName_value, testName_status, testName_unit}
    for tr in (linha.get("testResults") or []):
      test        = tr.get("test", {}) or {}
      translation = test.get("translation") or {}
      test_group  = test.get("testGroup", {}) or {}
      test_name   = translation.get("name") or test.get("id", "unknown")
      col         = test_name.strip().replace(" ", "_").replace("/", "_")

      base[f"{col}_value"]         = tr.get("resultValue")
      base[f"{col}_status"]        = tr.get("resultStatus")
      base[f"{col}_unit"]          = translation.get("unitOfMeasure")
      base[f"{col}_testresultsId"] = tr.get("id")           # id do resultado
      base[f"{col}_testId"]        = test.get("id")         # id do teste
      base[f"{col}_testType"]      = translation.get("name")
      base[f"{col}_testMethod"]    = translation.get("method")
      base[f"{col}_testName"]      = test_group.get("name")

    return base


# ── Paginação ────────────────────────────────────────────────────────────────
offset = 0
todos_registros = []

while True:
    print(f"Buscando offset {offset}...")
    resultados = buscar_dados(offset)
    if not resultados:
        break

    for item in resultados:
        todos_registros.append(processar_linha(item))

    offset += 50
    time.sleep(25)

Buscando offset 0...
Buscando offset 50...


##**4.2. DataFrame**

In [8]:
ordem_colunas = [
"sampleNumber",
"evaluation",
"inspectionAction",
"resultStatus",
"resultDate",
"receiptDate",
"registrationDate",
"dateSampled",
"equipmentSite",
"equipmentArea",
"equipmentSector",
"equipmentTag",
"equipmentFamily",
"equipmentMaker",
"equipmentModel",
"compartmentName",
"compartmentTypeName",
"oilType",
"oilViscosity",

"Visual_testId",
"Visual_testName",
"Visual_testType",
"Visual_testMethod",
"Visual_testresultsId",
"Visual_value",
"Visual_unit",
"Visual_status",

"PQ_Index_testId",
"PQ_Index_testName",
"PQ_Index_testType",
"PQ_Index_testMethod",
"PQ_Index_testresultsId",
"PQ_Index_value",
"PQ_Index_unit",
"PQ_Index_status",

"Oxidação-FTIR_testId",
"Oxidação-FTIR_testName",
"Oxidação-FTIR_testType",
"Oxidação-FTIR_testMethod",
"Oxidação-FTIR_testresultsId",
"Oxidação-FTIR_value",
"Oxidação-FTIR_unit",
"Oxidação-FTIR_status",

"Viscosidade_40°C_testId",
"Viscosidade_40°C_testName",
"Viscosidade_40°C_testType",
"Viscosidade_40°C_testMethod",
"Viscosidade_40°C_testresultsId",
"Viscosidade_40°C_value",
"Viscosidade_40°C_unit",
"Viscosidade_40°C_status",

"KF_Coulométrico_testId",
"KF_Coulométrico_testName",
"KF_Coulométrico_testType",
"KF_Coulométrico_testMethod",
"KF_Coulométrico_testresultsId",
"KF_Coulométrico_value",
"KF_Coulométrico_unit",
"KF_Coulométrico_status",

"TAN_Colorimétrico_testId",
"TAN_Colorimétrico_testName",
"TAN_Colorimétrico_testType",
"TAN_Colorimétrico_testMethod",
"TAN_Colorimétrico_testresultsId",
"TAN_Colorimétrico_value",
"TAN_Colorimétrico_unit",
"TAN_Colorimétrico_status",

"Cromo_testId",
"Cromo_testName",
"Cromo_testType",
"Cromo_testMethod",
"Cromo_testresultsId",
"Cromo_value",
"Cromo_unit",
"Cromo_status",

"Cobre_testId",
"Cobre_testName",
"Cobre_testType",
"Cobre_testMethod",
"Cobre_testresultsId",
"Cobre_value",
"Cobre_unit",
"Cobre_status",

"Vanádio_testId",
"Vanádio_testName",
"Vanádio_testType",
"Vanádio_testMethod",
"Vanádio_testresultsId",
"Vanádio_value",
"Vanádio_unit",
"Vanádio_status",

"Potássio_testId",
"Potássio_testName",
"Potássio_testType",
"Potássio_testMethod",
"Potássio_testresultsId",
"Potássio_value",
"Potássio_unit",
"Potássio_status",

"Estanho_testId",
"Estanho_testName",
"Estanho_testType",
"Estanho_testMethod",
"Estanho_testresultsId",
"Estanho_value",
"Estanho_unit",
"Estanho_status",

"Prata_testId",
"Prata_testName",
"Prata_testType",
"Prata_testMethod",
"Prata_testresultsId",
"Prata_value",
"Prata_unit",
"Prata_status",

"Molibdênio_testId",
"Molibdênio_testName",
"Molibdênio_testType",
"Molibdênio_testMethod",
"Molibdênio_testresultsId",
"Molibdênio_value",
"Molibdênio_unit",
"Molibdênio_status",

"Sódio_testId",
"Sódio_testName",
"Sódio_testType",
"Sódio_testMethod",
"Sódio_testresultsId",
"Sódio_value",
"Sódio_unit",
"Sódio_status",

"Zinco_testId",
"Zinco_testName",
"Zinco_testType",
"Zinco_testMethod",
"Zinco_testresultsId",
"Zinco_value",
"Zinco_unit",
"Zinco_status",

"Alumínio_testId",
"Alumínio_testName",
"Alumínio_testType",
"Alumínio_testMethod",
"Alumínio_testresultsId",
"Alumínio_value",
"Alumínio_unit",
"Alumínio_status",

"Manganês_testId",
"Manganês_testName",
"Manganês_testType",
"Manganês_testMethod",
"Manganês_testresultsId",
"Manganês_value",
"Manganês_unit",
"Manganês_status",

"Silício_testId",
"Silício_testName",
"Silício_testType",
"Silício_testMethod",
"Silício_testresultsId",
"Silício_value",
"Silício_unit",
"Silício_status",

"Magnésio_testId",
"Magnésio_testName",
"Magnésio_testType",
"Magnésio_testMethod",
"Magnésio_testresultsId",
"Magnésio_value",
"Magnésio_unit",
"Magnésio_status",

"Ferro_testId",
"Ferro_testName",
"Ferro_testType",
"Ferro_testMethod",
"Ferro_testresultsId",
"Ferro_value",
"Ferro_unit",
"Ferro_status",

"Níquel_testId",
"Níquel_testName",
"Níquel_testType",
"Níquel_testMethod",
"Níquel_testresultsId",
"Níquel_value",
"Níquel_unit",
"Níquel_status",

"Cádmio_testId",
"Cádmio_testName",
"Cádmio_testType",
"Cádmio_testMethod",
"Cádmio_testresultsId",
"Cádmio_value",
"Cádmio_unit",
"Cádmio_status",

"Bário_testId",
"Bário_testName",
"Bário_testType",
"Bário_testMethod",
"Bário_testresultsId",
"Bário_value",
"Bário_unit",
"Bário_status",

"Fósforo_testId",
"Fósforo_testName",
"Fósforo_testType",
"Fósforo_testMethod",
"Fósforo_testresultsId",
"Fósforo_value",
"Fósforo_unit",
"Fósforo_status",

"Cálcio_testId",
"Cálcio_testName",
"Cálcio_testType",
"Cálcio_testMethod",
"Cálcio_testresultsId",
"Cálcio_value",
"Cálcio_unit",
"Cálcio_status",

"Chumbo_testId",
"Chumbo_testName",
"Chumbo_testType",
"Chumbo_testMethod",
"Chumbo_testresultsId",
"Chumbo_value",
"Chumbo_unit",
"Chumbo_status",

"Boro_testId",
"Boro_testName",
"Boro_testType",
"Boro_testMethod",
"Boro_testresultsId",
"Boro_value",
"Boro_unit",
"Boro_status",

"Titânio_testId",
"Titânio_testName",
"Titânio_testType",
"Titânio_testMethod",
"Titânio_testresultsId",
"Titânio_value",
"Titânio_unit",
"Titânio_status",

"Classe_ISO_testId",
"Classe_ISO_testName",
"Classe_ISO_testType",
"Classe_ISO_testMethod",
"Classe_ISO_testresultsId",
"Classe_ISO_value",
"Classe_ISO_unit",
"Classe_ISO_status",

"SAE_4µm_testId",
"SAE_4µm_testName",
"SAE_4µm_testType",
"SAE_4µm_testMethod",
"SAE_4µm_testresultsId",
"SAE_4µm_value",
"SAE_4µm_unit",
"SAE_4µm_status",

"SAE_6µm_testId",
"SAE_6µm_testName",
"SAE_6µm_testType",
"SAE_6µm_testMethod",
"SAE_6µm_testresultsId",
"SAE_6µm_value",
"SAE_6µm_unit",
"SAE_6µm_status",

"SAE_14µm_testId",
"SAE_14µm_testName",
"SAE_14µm_testType",
"SAE_14µm_testMethod",
"SAE_14µm_testresultsId",
"SAE_14µm_value",
"SAE_14µm_unit",
"SAE_14µm_status",

"SAE_21µm_testId",
"SAE_21µm_testName",
"SAE_21µm_testType",
"SAE_21µm_testMethod",
"SAE_21µm_testresultsId",
"SAE_21µm_value",
"SAE_21µm_unit",
"SAE_21µm_status",

"SAE_38µm_testId",
"SAE_38µm_testName",
"SAE_38µm_testType",
"SAE_38µm_testMethod",
"SAE_38µm_testresultsId",
"SAE_38µm_value",
"SAE_38µm_unit",
"SAE_38µm_status",

"SAE_70µm_testId",
"SAE_70µm_testName",
"SAE_70µm_testType",
"SAE_70µm_testMethod",
"SAE_70µm_testresultsId",
"SAE_70µm_value",
"SAE_70µm_unit",
"SAE_70µm_status",

">4_testId",
">4_testName",
">4_testType",
">4_testMethod",
">4_testresultsId",
">4_value",
">4_unit",
">4_status",

">6_testId",
">6_testName",
">6_testType",
">6_testMethod",
">6_testresultsId",
">6_value",
">6_unit",
">6_status",

">14_testId",
">14_testName",
">14_testType",
">14_testMethod",
">14_testresultsId",
">14_value",
">14_unit",
">14_status"
]

tb_results = pd.DataFrame(todos_registros)

tb_results = tb_results.reindex(columns=ordem_colunas)

In [9]:
display(tb_results)

,sampleNumber,evaluation,inspectionAction,resultStatus,resultDate,receiptDate,registrationDate,dateSampled,equipmentSite,equipmentArea,...,>6_unit,>6_status,>14_testId,>14_testName,>14_testType,>14_testMethod,>14_testresultsId,>14_value,>14_unit,>14_status
0,2600089665,Os resultados são aceitáveis ​​quanto a taxa ...,None,NORMAL,2026-05-12T20:03:48Z,2026-05-07T15:00:00Z,2026-05-08T16:17:49Z,2026-03-30T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,,...,part/10mL,None,802,Contaminação,>14,ASTM D7647/ ISO4406,184985202,604,part/10mL,None
1,2600309048,Condição do Lubrificante: A amostra apresenta...,Recomendamos que seja efetuada a micro filtra...,SEVERE,2026-05-12T19:53:34Z,2026-05-11T15:22:00Z,2026-05-08T12:31:14Z,2026-05-03T03:00:00Z,SAMARCO MINERACAO - GERMANO - MARIANA MG,BRITAGEM III,...,part/10mL,SEVERE,802,Contaminação,>14,ASTM D7647/ ISO4406,185129965,90132,part/10mL,SEVERE
2,2600183112,Condição do Lubrificante: A amostra apresenta ...,Recomendamos avaliar entre tratar ou efetuar ...,SEVERE,2026-05-12T18:20:22Z,2026-05-07T15:00:00Z,2026-05-05T16:47:55Z,2026-04-14T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,MISTURA,...,part/10mL,SEVERE,802,Contaminação,>14,ASTM D7647/ ISO4406,185082717,49,part/10mL,None
3,2600089654,Condição do Lubrificante: A amostra apresenta ...,Recomendamos que seja efetuada a micro filtra...,SEVERE,2026-05-12T15:12:38Z,2026-05-07T15:00:00Z,2026-05-05T16:49:28Z,2026-03-31T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,CALCÁRIO,...,part/10mL,SEVERE,802,Contaminação,>14,ASTM D7647/ ISO4406,185033825,1763,part/10mL,SEVERE
4,2600183308,Os resultados não condizem com o histórico do...,Verificar o procedimento de coleta e o corret...,SEVERE,2026-05-12T14:16:45Z,2026-05-07T15:00:00Z,2026-05-05T16:49:41Z,2026-04-14T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,CARVÃO,...,part/10mL,None,802,Contaminação,>14,ASTM D7647/ ISO4406,185033875,277,part/10mL,None
5,2600088649,Os resultados são aceitáveis ​​quanto a taxa ...,None,NORMAL,2026-05-12T14:11:54Z,2026-05-07T15:00:00Z,2026-05-08T19:08:05Z,None,SAMARCO MINERACAO - UBU - ANCHIETA ES,FORNO,...,part/10mL,None,802,Contaminação,>14,ASTM D7647/ ISO4406,184985214,152,part/10mL,None
6,2600037441,Condição do Lubrificante: A amostra apresenta...,Recomendamos que seja efetuada a micro filtra...,SEVERE,2026-05-12T14:11:25Z,2026-05-07T15:00:00Z,2026-05-05T16:48:17Z,2026-03-16T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,PELOTAMENTO,...,part/10mL,SEVERE,802,Contaminação,>14,ASTM D7647/ ISO4406,185034024,144,part/10mL,None
7,2600183068,Condição do Lubrificante: A amostra apresenta ...,Recomendamos avaliar entre tratar ou efetuar ...,SEVERE,2026-05-12T14:10:45Z,2026-05-07T15:00:00Z,2026-05-05T16:48:00Z,2026-04-14T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,MISTURA,...,part/10mL,None,802,Contaminação,>14,ASTM D7647/ ISO4406,185034065,1609,part/10mL,None
8,2600088529,Condição do lubrificante: Apresenta condições ...,Recomendamos que seja feita nova coleta dentr...,ABNORMAL,2026-05-12T14:09:32Z,2026-05-07T15:00:00Z,2026-05-05T16:47:36Z,2026-03-12T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,PENEIRAMENTO,...,part/10mL,None,802,Contaminação,>14,ASTM D7647/ ISO4406,184997079,-,part/10mL,None
9,2600088624,Os resultados são aceitáveis ​​quanto a taxa ...,None,NORMAL,2026-05-12T14:06:50Z,2026-05-07T15:00:00Z,2026-05-05T16:47:34Z,2026-04-09T00:00:00Z,SAMARCO MINERACAO - UBU - ANCHIETA ES,PENEIRAMENTO,...,part/10mL,None,802,Contaminação,>14,ASTM D7647/ ISO4406,185033904,1570,part/10mL,None


In [ ]:
#download excel
tb_results.to_excel("cda_lub_ext_als_results.xlsx", index=False)

##**4.3. Carga no Sheets**

In [10]:
# Nome da planilha e aba
nome_da_planilha = "cda_lub_ext_als_results"
nome_da_aba = "Sheet1"

# Se a busca não retornar dados, encerra o processo de envio
if tb_results.empty:
    print("Nenhum dado novo encontrado na API para ontem. Nada a adicionar.")

else:
    # Abre a planilha
    planilha = gc.open(nome_da_planilha)
    aba = planilha.worksheet(nome_da_aba)

    # Lê os dados atuais da planilha como DataFrame
    df_existente = get_as_dataframe(
        aba,
        evaluate_formulas=True,
        dtype=str
    ).dropna(how="all")

    # Normaliza nomes de colunas (evita espaço invisível no header)
    if not df_existente.empty:
        df_existente.columns = df_existente.columns.str.strip()

    # Se não existir a coluna sampleNumber ainda, cria estrutura mínima
    if df_existente.empty or "sampleNumber" not in df_existente.columns:
        df_existente = pd.DataFrame(columns=["sampleNumber"])

    else:
        df_existente = df_existente.dropna(subset=["sampleNumber"])

    # Garante que sampleNumber seja string para comparação
    tb_results["sampleNumber"] = tb_results["sampleNumber"].astype(str)
    df_existente["sampleNumber"] = df_existente["sampleNumber"].astype(str)

    # Sincroniza header da planilha se houver diferença de colunas
    colunas_planilha = list(df_existente.columns)
    colunas_df = list(tb_results.columns)

    if colunas_planilha != colunas_df:
        print("Estrutura da planilha desatualizada — recriando header...")
        aba.clear()
        set_with_dataframe(
            aba,
            tb_results,
            include_index=False,
            include_column_header=True,
            resize=True
        )
        print("Planilha sincronizada com sucesso.")

    else:
        # Filtra apenas os novos registros
        novos_registros = tb_results[
            ~tb_results["sampleNumber"].isin(df_existente["sampleNumber"])
        ]

        # Se houver novos dados, adiciona ao final da aba
        if not novos_registros.empty:
            proxima_linha = len(df_existente) + 2  # +1 header, +1 próxima linha

            set_with_dataframe(
                aba,
                novos_registros,
                row=proxima_linha,
                include_column_header=False
            )

            print(f"{len(novos_registros)} novas linhas adicionadas ao Google Sheets!")
        else:
            print("Nenhuma nova linha para adicionar — todos os dados já estão presentes.")

39 novas linhas adicionadas ao Google Sheets!


# **5. PSEUDOCÓDIGO**

# CDA LUB EXT ALS (Análise de Lubrificantes)

---

## 1. CONFIGURAÇÕES INICIAIS

```
DEFINIR credentials = { username, password }

DEFINIR url_base          = "https://api.s360web.com"
DEFINIR url_login         = url_base + "/api/login"
DEFINIR url_samplelist    = url_base + "/api/v1/amostra/list"
DEFINIR url_sampleresults = url_base + "/api/v1/sampleResult/search"

AUTENTICAR no Google Sheets via OAuth (gspread + google.auth)
```

---

## 2. AUTENTICAÇÃO

```
FUNÇÃO obter_token():
    POST para url_login
        headers = { Content-Type: application/json }
        body    = JSON(credentials)

    access_token = resposta.json()["access_token"]
    RETORNAR access_token

access_token = obter_token()

header = {
    Authorization : "Bearer {access_token}",
    Content-Type  : "application/json"
}
```

---

## 3. STATUS DE AMOSTRAS

### 3.1. Busca paginada com paralelismo

```
FUNÇÃO fetch_pagina(pagina):
    TENTAR:
        filters = {
            maximoPorPagina : 100,
            numeroPagina    : pagina,
            situacao        : ["EM_PROCESSAMENTO", "COLETADA",
                               "SEGREGADA", "FINALIZADA"]
        }
        POST para url_samplelist com filters
        RETORNAR resposta["resultados"]
    EXCETO erro:
        registrar erro
        RETORNAR []

// Busca página 1 para descobrir total de páginas
POST url_samplelist com filtros (pagina=1)
total_paginas   = resposta["totalPaginas"]
resultados_totais = resposta["resultados"]

// Busca páginas restantes em paralelo (5 threads)
paginas_restantes = [2, 3, ..., total_paginas]
PARALELAMENTE (ThreadPoolExecutor, max_workers=5):
    PARA CADA pagina EM paginas_restantes:
        executar fetch_pagina(pagina)
        resultados_totais += resultado

IMPRIMIR total de registros coletados
```

### 3.2. Estrutura e organização

```
// Expande campos aninhados (cliente, obra, etc.) até 2 níveis
tb_status = json_normalize(resultados_totais, sep="_", max_level=2)
```

### 3.3. Carga no Sheets

```
ABRIR planilha "cda_lub_ext_als_status" → aba "Sheet1"
LIMPAR aba
ESCREVER tb_status completo na aba
```

---

## 4. RESULTADO DE AMOSTRAS

### 4.1. Funções de busca e transformação

#### Busca paginada por offset
```
FUNÇÃO buscar_dados(offset):
    ontem = data_atual - 1 dia (formato ISO)

    filters = {
        sinceResultDate : ontem,
        untilResultDate : ontem,
        offset          : offset,
        max             : 50,
        order           : "resultDate",
        sort            : "desc"
    }

    POST para url_sampleresults com filters

    TENTAR:
        RETORNAR resposta["results"]
    EXCETO erro:
        registrar erro
        RETORNAR []
```

#### Processamento e pivotagem de cada linha
```
FUNÇÃO processar_linha(linha):

    // Extrair objetos aninhados com fallback para {}
    validResult      = linha.validResult      ou {}
    equipment        = linha.equipment        ou {}
    collectionData   = linha.collectionData   ou {}
    oil              = collectionData.oil     ou {}
    compartment      = linha.compartment      ou {}
    site             = equipment.site         ou {}
    family           = equipment.family       ou {}
    maker            = equipment.maker        ou {}
    oil_viscosity    = oil.viscosity          ou {}
    oil_manufacturer = oil.manufacturer       ou {}
    compartment_type = compartment.type       ou {}

    // Campos base da amostra
    base = {
        sampleNumber      : linha.sampleNumber,
        evaluation        : validResult.evaluation,
        inspectionAction  : validResult.inspectionAction,
        resultStatus      : validResult.resultStatus,
        equipmentTag      : equipment.tag,
        equipmentFamily   : family.name,
        equipmentMaker    : maker.name,
        equipmentModel    : equipment.model,
        equipmentSite     : site.name,
        equipmentArea     : equipment.area,
        equipmentSector   : equipment.sector,
        compartmentName   : compartment.name,
        compartmentTypeName: compartment_type.name,
        registrationDate  : collectionData.registrationDate,
        dateSampled       : collectionData.dateSampled,
        oilViscosity      : oil_viscosity.name,
        oilType           : oil_manufacturer.name,
        receiptDate       : linha.receiptDate,
        resultDate        : linha.resultDate
    }

    // Pivotar testResults → colunas dinâmicas por tipo de teste
    // Cada teste gera 8 colunas: {col}_value, _status, _unit,
    //                             _testresultsId, _testId, _testType,
    //                             _testMethod, _testName
    PARA CADA tr EM (linha.testResults ou []):
        test        = tr.test        ou {}
        translation = test.translation ou {}
        test_group  = test.testGroup   ou {}
        test_name   = translation.name ou test.id ou "unknown"

        // Normalizar nome para nome de coluna seguro
        col = test_name.strip().replace(" ", "_").replace("/", "_")

        base["{col}_value"]         = tr.resultValue
        base["{col}_status"]        = tr.resultStatus
        base["{col}_unit"]          = translation.unitOfMeasure
        base["{col}_testresultsId"] = tr.id
        base["{col}_testId"]        = test.id
        base["{col}_testType"]      = translation.name
        base["{col}_testMethod"]    = translation.method
        base["{col}_testName"]      = test_group.name

    RETORNAR base
```

#### Loop de paginação com throttle
```
offset = 0
todos_registros = []

LOOP:
    resultados = buscar_dados(offset)

    SE resultados vazio: SAIR loop

    PARA CADA item EM resultados:
        todos_registros += processar_linha(item)

    offset += 50
    aguardar 25 segundos   // respeitar rate limit da API

df_samples = DataFrame(todos_registros)
SALVAR Excel "cda_lub_ext_als_results.xlsx"
```

---

### 4.2. Estrutura e organização do DataFrame

```
tb_results = DataFrame(todos_registros)

// Reindexar com ordem de colunas padronizada:
// ─ Campos base (identificação, datas, equipamento, compartimento, óleo)
// ─ Testes laboratoriais, cada um com 8 sub-colunas:
//   Visual, PQ_Index, Oxidação-FTIR, Viscosidade_40°C,
//   KF_Coulométrico, TAN_Colorimétrico,
//   Metais de desgaste:
//     Cromo, Cobre, Vanádio, Potássio, Estanho, Prata,
//     Molibdênio, Sódio, Zinco, Alumínio, Manganês,
//     Silício, Magnésio, Ferro, Níquel, Cádmio, Bário,
//     Fósforo, Cálcio, Chumbo, Boro, Titânio
//   Contagem de partículas:
//     Classe_ISO,
//     SAE_4µm, SAE_6µm, SAE_14µm, SAE_21µm, SAE_38µm, SAE_70µm,
//     >4, >6, >14

tb_results = tb_results.reindex(columns=ordem_colunas)
SALVAR Excel "cda_lub_ext_als_results.xlsx"
```

---

### 4.3. Carga incremental no Sheets

```
ABRIR planilha "cda_lub_ext_als_results" → aba "Sheet1"

SE tb_results vazio:
    IMPRIMIR "Nenhum dado novo" e encerrar

// Ler dados existentes na aba
df_existente = ler aba como DataFrame (dtype=str)
normalizar nomes de colunas (strip)

SE df_existente vazio OU sem coluna "sampleNumber":
    df_existente = DataFrame vazio com coluna "sampleNumber"
SENÃO:
    remover linhas onde sampleNumber é nulo

// Garantir tipo string para comparação
tb_results["sampleNumber"]   = string
df_existente["sampleNumber"] = string

// Verificar se a estrutura de colunas diverge
SE colunas do df_existente != colunas do tb_results:
    // Estrutura desatualizada → recriar aba completa
    LIMPAR aba
    ESCREVER tb_results inteiro (com cabeçalho)
    IMPRIMIR "Planilha sincronizada com sucesso"

SENÃO:
    // Estrutura igual → inserir apenas registros novos
    novos_registros = tb_results
        ONDE sampleNumber NÃO EM df_existente["sampleNumber"]

    SE novos_registros não vazio:
        proxima_linha = len(df_existente) + 2  // +1 header, +1 offset
        INSERIR novos_registros a partir de proxima_linha (sem cabeçalho)
        IMPRIMIR "{N} novas linhas adicionadas"

    SENÃO:
        IMPRIMIR "Nenhuma nova linha — tudo já está presente"
```

---

## FLUXO GERAL (visão macro)

```
INÍCIO
│
├─ Configurar credenciais e URLs da API S360
├─ Autenticar: POST /api/login → obter access_token → montar header
├─ Autenticar no Google Sheets
│
├─ [3] STATUS DE AMOSTRAS
│       └─ POST /amostra/list (pág. 1) → descobrir total_paginas
│       └─ buscar páginas restantes em paralelo (5 threads)
│       └─ json_normalize (expande campos aninhados)
│       └─ Sheets "cda_lub_ext_als_status" (limpar + escrever)
│
└─ [4] RESULTADO DE AMOSTRAS
        └─ Loop paginado por offset (50 por página)
        │   └─ POST /sampleResult/search (ontem → ontem)
        │   └─ processar_linha(): extrair campos base +
        │       pivotar testResults → colunas {teste}_value/status/unit/...
        │   └─ aguardar 25s entre páginas (rate limit)
        │
        ├─ Reindexar tb_results com ordem de colunas padrão
        │   (identificação → datas → equipamento → testes laboratoriais
        │    → metais de desgaste → contagem de partículas)
        ├─ Excel "cda_lub_ext_als_results.xlsx"
        │
        └─ Carga incremental no Sheets "cda_lub_ext_als_results":
            ├─ SE estrutura diverge → recriar aba completa
            └─ SE estrutura igual → inserir apenas sampleNumbers novos

FIM
```

---

## PADRÃO DE COLUNAS POR TESTE LABORATORIAL

```
// Para cada tipo de teste, são geradas 8 colunas:
{NomeTeste}_value         → valor medido
{NomeTeste}_status        → status do resultado (Normal, Atenção, Crítico...)
{NomeTeste}_unit          → unidade de medida
{NomeTeste}_testresultsId → ID do registro de resultado
{NomeTeste}_testId        → ID do tipo de teste
{NomeTeste}_testType      → nome traduzido do teste
{NomeTeste}_testMethod    → método analítico utilizado
{NomeTeste}_testName      → grupo do teste (família analítica)

// Testes contemplados (30 análises):
// Físico-químicos : Visual, PQ_Index, Oxidação-FTIR,
//                  Viscosidade_40°C, KF_Coulométrico, TAN_Colorimétrico
// Metais (ICP)    : Cromo, Cobre, Vanádio, Potássio, Estanho, Prata,
//                  Molibdênio, Sódio, Zinco, Alumínio, Manganês,
//                  Silício, Magnésio, Ferro, Níquel, Cádmio,
//                  Bário, Fósforo, Cálcio, Chumbo, Boro, Titânio
// Partículas      : Classe_ISO, SAE_4µm, SAE_6µm, SAE_14µm,
//                  SAE_21µm, SAE_38µm, SAE_70µm, >4, >6, >14
```